# jarvis — case study

Paper-trading run analysis. Reads `data/trades.db` directly. Re-run any time after the agent has been trading for at least a few cycles.

**Guiding question:** did an LLM-in-the-loop decision process beat a naive buy-and-hold baseline on the same assets over the same window, net of paper slippage?

## 1. Load data

In [ ]:
import sqlite3, json, pathlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 4)

DB = pathlib.Path('../data/trades.db')
assert DB.exists(), f'no DB at {DB}'
conn = sqlite3.connect(DB)

positions = pd.read_sql('SELECT * FROM positions', conn, parse_dates=['opened_at', 'closed_at'])
decisions = pd.read_sql('SELECT * FROM decisions', conn, parse_dates=['timestamp'])
snapshots = pd.read_sql('SELECT * FROM candles_snapshots', conn, parse_dates=['timestamp'])
account = pd.read_sql('SELECT * FROM account', conn).iloc[0]

print(f'positions rows:     {len(positions)}')
print(f'decisions rows:     {len(decisions)}')
print(f'snapshots rows:     {len(snapshots)}')
print(f'starting balance:   ${account.initial_balance:,.2f}')
print(f'current balance:    ${account.balance:,.2f}')
print(f'data range:         {decisions.timestamp.min()} -> {decisions.timestamp.max()}')

## 2. Trade summary stats

Basic PnL math on closed trades.

In [ ]:
closed = positions[positions.status == 'closed'].copy()
closed['wins'] = closed.realized_pnl > 0

wins = closed[closed.wins]
losses = closed[~closed.wins]

avg_win = wins.realized_pnl.mean() if len(wins) else 0
avg_loss = losses.realized_pnl.mean() if len(losses) else 0
profit_factor = (wins.realized_pnl.sum() / -losses.realized_pnl.sum()) if len(losses) and losses.realized_pnl.sum() != 0 else float('inf')

summary = pd.Series({
    'total_closed':     len(closed),
    'win_rate':         len(wins) / len(closed) if len(closed) else 0,
    'total_pnl':        closed.realized_pnl.sum() if len(closed) else 0,
    'avg_win':          avg_win,
    'avg_loss':         avg_loss,
    'profit_factor':    profit_factor,
    'largest_win':      closed.realized_pnl.max() if len(closed) else 0,
    'largest_loss':     closed.realized_pnl.min() if len(closed) else 0,
}).round(4)
summary

## 3. Per-asset breakdown

In [ ]:
if len(closed):
    per_asset = closed.groupby('asset').agg(
        trades=('id', 'count'),
        pnl=('realized_pnl', 'sum'),
        win_rate=('wins', 'mean'),
        avg_pnl=('realized_pnl', 'mean'),
    ).round(4).sort_values('pnl', ascending=False)
    display(per_asset)
else:
    print('no closed trades yet')

## 4. Equity curve

Reconstruct balance over time from realized PnL + latest account value.

In [ ]:
if len(closed):
    eq = closed.sort_values('closed_at').copy()
    eq['cum_pnl'] = eq.realized_pnl.cumsum()
    eq['equity'] = account.initial_balance + eq.cum_pnl
    plt.plot(eq.closed_at, eq.equity, lw=1.5)
    plt.axhline(account.initial_balance, color='grey', ls='--', lw=0.7, label='starting balance')
    plt.title('equity curve (realized only)')
    plt.ylabel('account value (USD)')
    plt.legend(); plt.tight_layout()
else:
    print('need at least one closed trade for an equity curve')

## 5. Drawdown & max drawdown

In [ ]:
if len(closed):
    peaks = eq.equity.cummax()
    dd = (eq.equity - peaks) / peaks * 100
    plt.plot(eq.closed_at, dd, lw=1, color='red')
    plt.fill_between(eq.closed_at, dd, 0, color='red', alpha=0.15)
    plt.title(f'drawdown — max {dd.min():.2f}%')
    plt.ylabel('% from peak')
    plt.tight_layout()

## 6. LLM decision quality

Rough heuristic: tag each closed trade with keywords from the LLM's rationale at entry, then see which keywords correlate with winners vs losers.

This is a quick pass — a proper study would use NLP, but keyword buckets are enough to spot obvious patterns (e.g. "MACD cross" beats "RSI oversold").

In [ ]:
keywords = ['macd', 'rsi', 'ema', 'trend', 'breakout', 'oversold', 'overbought',
            'funding', 'support', 'resistance', 'volume', 'momentum', 'reversal']

def tag(text):
    t = (text or '').lower()
    return [k for k in keywords if k in t]

if len(closed) and len(decisions):
    ent = decisions[decisions.action.isin(['buy','sell'])].sort_values('timestamp')
    closed_ent = pd.merge_asof(
        closed.sort_values('opened_at')[['asset', 'opened_at', 'realized_pnl']],
        ent[['asset', 'timestamp', 'rationale']],
        left_on='opened_at', right_on='timestamp', by='asset',
        direction='backward', tolerance=pd.Timedelta('1h')
    )
    closed_ent['keywords'] = closed_ent.rationale.apply(tag)
    kw_rows = []
    for _, r in closed_ent.iterrows():
        for k in r.keywords:
            kw_rows.append({'keyword': k, 'pnl': r.realized_pnl})
    kw = pd.DataFrame(kw_rows)
    if len(kw):
        stats = kw.groupby('keyword').agg(
            trades=('pnl','count'),
            win_rate=('pnl', lambda s: (s > 0).mean()),
            avg_pnl=('pnl','mean')
        ).round(3).sort_values('avg_pnl', ascending=False)
        display(stats)
    else:
        print('no keyword matches yet')
else:
    print('not enough data yet')

## 7. Baseline: equal-weight buy-and-hold

Compare the agent's realized PnL against what would have happened if we had simply split the starting balance evenly across the same assets and held. Uses `candles_snapshots` entry/exit prices for the agent's assets across the full run window.

In [ ]:
if len(snapshots):
    assets = snapshots.asset.unique().tolist()
    first = snapshots.sort_values('timestamp').groupby('asset').first().reset_index()
    last = snapshots.sort_values('timestamp').groupby('asset').last().reset_index()
    base_alloc = account.initial_balance / len(assets)
    bh = pd.DataFrame({
        'asset': assets,
        'first_px': first.set_index('asset').loc[assets, 'close_price'].values,
        'last_px': last.set_index('asset').loc[assets, 'close_price'].values,
    })
    bh['return_pct'] = (bh.last_px - bh.first_px) / bh.first_px * 100
    bh['bh_pnl'] = base_alloc * bh.return_pct / 100
    display(bh.round(3))
    print(f'equal-weight buy-and-hold total pnl: ${bh.bh_pnl.sum():,.2f}')
    print(f'agent realized pnl:                  ${closed.realized_pnl.sum() if len(closed) else 0:,.2f}')
else:
    print('no snapshots yet')

## 8. Honest conclusion

[FILL IN AFTER 4-WEEK RUN]

- **What worked.** _e.g._ The hysteresis rule prevented the early churn I saw on day 1–2. MACD-flagged entries had the highest win rate.
- **What didn't.** _e.g._ RSI-oversold reversal trades were systematic losers — the LLM took them even when the 4h trend was still down. That matches the classical lit, which is a useful negative result.
- **What I'd change.** _e.g._ Feed the LLM a regime label (trending vs chop) so it can suppress mean-reversion trades in trending regimes.
- **Caveats.** Paper slippage is fixed at 0.05%. No funding cost. Market was [trending/ranging/choppy] in this window — results may not generalize.